In [1]:
import cv2
import mediapipe as mp
import pandas as pd
import os

In [3]:
def process_dance_pose(video_path, output_csv, debug_video_path):
    mp_pose = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    # Setting model_complexity to 2 for highest accuracy (slower but better)
    pose = mp_pose.Pose(static_image_mode=False, 
                        model_complexity=2, 
                        min_detection_confidence=0.6)
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Setup Debug Video Writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_video = cv2.VideoWriter(debug_video_path, fourcc, fps, (width, height))
    
    data = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        results = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        if results.pose_landmarks:
            # Draw skeleton on frame for verification
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
            
            # Extract data
            row = [frame_idx]
            for lm in results.pose_landmarks.landmark:
                # We save X, Y, Z and the Confidence (Visibility)
                row.extend([lm.x, lm.y, lm.z, lm.visibility])
            data.append(row)
        
        out_video.write(frame)
        frame_idx += 1
        if frame_idx % 100 == 0: print(f"Processed {frame_idx} frames...")

    # Save CSV
    cols = ['frame']
    for i in range(33): cols.extend([f'x{i}', f'y{i}', f'z{i}', f'conf{i}'])
    pd.DataFrame(data, columns=cols).to_csv(output_csv, index=False)
    
    cap.release()
    out_video.release()
    print("Done! Check 'verify_pose.mp4' to see if coordinates are correct.")

if __name__ == "__main__":
    process_dance_pose('dance_final.mp4', 'dance_dataset.csv', 'verify_pose.mp4')

I0000 00:00:1776020338.783402   31908 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776020338.789986   33216 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: Mesa Intel(R) UHD Graphics 630 (CFL GT2)


Processed 100 frames...
Processed 200 frames...
Processed 300 frames...
Processed 400 frames...
Processed 500 frames...
Processed 600 frames...
Processed 700 frames...
Processed 800 frames...
Processed 900 frames...
Processed 1000 frames...
Processed 1100 frames...
Processed 1200 frames...
Processed 1300 frames...
Processed 1400 frames...
Processed 1500 frames...
Processed 1600 frames...
Processed 1700 frames...
Processed 1800 frames...
Processed 1900 frames...
Processed 2000 frames...
Processed 2100 frames...
Processed 2200 frames...
Processed 2300 frames...
Processed 2400 frames...
Processed 2500 frames...
Processed 2600 frames...
Processed 2700 frames...
Processed 2800 frames...
Processed 2900 frames...
Processed 3000 frames...
Processed 3100 frames...
Processed 3200 frames...
Processed 3300 frames...
Processed 3400 frames...
Processed 3500 frames...
Processed 3600 frames...
Processed 3700 frames...
Processed 3800 frames...
Processed 3900 frames...
Processed 4000 frames...
Processed

In [ ]:
''' 
Data filtering
'''


def create_final_dataset(csv_path, output_csv):
    # Check if input exists
    if not os.path.exists(csv_path):
        print(f"Error: {csv_path} not found.")
        return

    df = pd.read_csv(csv_path)
    
    # 1. Calculate Filters
    df['ankle_vis'] = (df['conf27'] + df['conf28']) / 2
    df['avg_leg_len'] = (abs(df['y27'] - df['y23']) + abs(df['y28'] - df['y24'])) / 2

    # 2. Filter
    mask = (df['ankle_vis'] > 0.65) & (df['avg_leg_len'] > 0.18)
    gold_df = df[mask].copy()
    
    # 3. Clean up columns
    gold_df = gold_df.drop(columns=['ankle_vis', 'avg_leg_len'])
    
    # 4. FORCE DELETE AND SAVE
    if os.path.exists(output_csv):
        os.remove(output_csv) # Delete the old 13k file
        print(f"Old {output_csv} deleted.")

    gold_df.to_csv(output_csv, index=False)
    
    # 5. VERIFY DISK WRITE
    actual_rows = len(pd.read_csv(output_csv))
    print(f"\n--- VERIFICATION ---")
    print(f"Original Rows: {len(df)}")
    print(f"Memory Check (Gold): {len(gold_df)}")
    print(f"Disk Check (Final File): {actual_rows}")
    
    if actual_rows == len(gold_df):
        print("SUCCESS: The file on disk is now filtered.")
    else:
        print("ERROR: File mismatch. Check folder permissions.")

if __name__ == "__main__":
    create_final_dataset('dance_dataset.csv', 'final_dance_dataset.csv')